#TAREA 3
## Leonardo Garcia Muñoz

- Usar dataframes de pyspark para manipular datos

In [ ]:
#Primero creamos una sesion de pyspark y leemos los datos (que ya bajamos de la api referenciada y mostrada en la tarea anterior)

In [2]:
from pyspark.sql import SparkSession
from pyspark import SparkContext

spark = SparkSession.builder \
    .appName("Tarea3BIGDATA") \
    .getOrCreate()

In [3]:
df = spark.read.csv(
    "/content/youtube_bigdata_comments_dataset_2.csv",
    header=True,
    inferSchema=False,
    multiLine=True,
    quote='"',
    escape='"',
    mode="PERMISSIVE")

In [4]:
df.show(5)
df.printSchema()

+--------------------+--------------+-----------+--------------------+--------------------+--------------------+-----+--------------------+-----------+
|          channel_id| channel_title|   video_id|         video_title|              author|             comment|likes|        published_at|reply_count|
+--------------------+--------------+-----------+--------------------+--------------------+--------------------+-----+--------------------+-----------+
|UCRZpxmNB22q_jXcL...|Conversaciones|IffJxvWNUiM|Contactar a Leo M...|@omarvalenteplaza...|mañana nuevo capi...|    0|2026-03-08T22:32:07Z|          0|
|UCRZpxmNB22q_jXcL...|Conversaciones|cxrvVgQ4bt4|Así se LOGRÓ el P...| @jeffersonsilva1089|Soy de Nicaragua ...|    0|2026-03-08T19:13:35Z|          0|
|UCRZpxmNB22q_jXcL...|Conversaciones|cxrvVgQ4bt4|Así se LOGRÓ el P...|  @JorgeDeLara-xg3uw|Adrian solo fue d...|    0|2026-03-08T16:09:53Z|          0|
|UCRZpxmNB22q_jXcL...|Conversaciones|cxrvVgQ4bt4|Así se LOGRÓ el P...|@miguelangelherre.

- Realizar manipulación de filas y columnas en los datos elegidos
  - Modificar datos
  - Agregar nuevas columnas calculadas
  - Filtrar resultados

In [5]:
#Primero modificamos los datos
from pyspark.sql.functions import col, lower, trim

df = df.withColumn("comment", lower(trim(col("comment")))) #con trim quitamos los espacios como la funcion trim de sql
#mientras que con lower mandamos todas las letras a minusculas

In [6]:
df.show(5)#ojo en la columna comment comparandola con la de arriba si se notan ejemplos de mayusculas que volvimos minusculas

+--------------------+--------------+-----------+--------------------+--------------------+--------------------+-----+--------------------+-----------+
|          channel_id| channel_title|   video_id|         video_title|              author|             comment|likes|        published_at|reply_count|
+--------------------+--------------+-----------+--------------------+--------------------+--------------------+-----+--------------------+-----------+
|UCRZpxmNB22q_jXcL...|Conversaciones|IffJxvWNUiM|Contactar a Leo M...|@omarvalenteplaza...|mañana nuevo capi...|    0|2026-03-08T22:32:07Z|          0|
|UCRZpxmNB22q_jXcL...|Conversaciones|cxrvVgQ4bt4|Así se LOGRÓ el P...| @jeffersonsilva1089|soy de nicaragua ...|    0|2026-03-08T19:13:35Z|          0|
|UCRZpxmNB22q_jXcL...|Conversaciones|cxrvVgQ4bt4|Así se LOGRÓ el P...|  @JorgeDeLara-xg3uw|adrian solo fue d...|    0|2026-03-08T16:09:53Z|          0|
|UCRZpxmNB22q_jXcL...|Conversaciones|cxrvVgQ4bt4|Así se LOGRÓ el P...|@miguelangelherre.

In [7]:
#Despues Agregamos nuevas columnas calculadas
#No se me ocurrio mas que agregar una columna que nos traiga la longitud del comentario
from pyspark.sql.functions import length,to_date,when, hour

df = df.withColumn("longitud_caracteres", length(col("comment")))

#Tambien otra que clasifique que tan popular es el comentario en interacciones/likes
df = df.withColumn(
    "popularidad",when(col("likes") > 1300, "viral").when(col("likes") > 130, "alta").when(col("likes") > 13, "media").otherwise("baja"))
#Tome el 13 como media, porque en la tarea anterior obtuvimos una media de 13 likes por comentario

#y un poco rebuscado pero una columna que traiga la fecha en formato solo date (quitar la hora/que deje de ser como timestamp)
df = df.withColumn("fecha_limpia", to_date(col("published_at")))
#y otra que traiga la hora limpia de la publicacion del comment
df = df.withColumn("hora_publicacion", hour(col("published_at")))

In [8]:
df.show(15)#observamos las nuevas columnas

+--------------------+--------------+-----------+--------------------+--------------------+--------------------+-----+--------------------+-----------+-------------------+-----------+------------+----------------+
|          channel_id| channel_title|   video_id|         video_title|              author|             comment|likes|        published_at|reply_count|longitud_caracteres|popularidad|fecha_limpia|hora_publicacion|
+--------------------+--------------+-----------+--------------------+--------------------+--------------------+-----+--------------------+-----------+-------------------+-----------+------------+----------------+
|UCRZpxmNB22q_jXcL...|Conversaciones|IffJxvWNUiM|Contactar a Leo M...|@omarvalenteplaza...|mañana nuevo capi...|    0|2026-03-08T22:32:07Z|          0|                 39|       baja|  2026-03-08|              22|
|UCRZpxmNB22q_jXcL...|Conversaciones|cxrvVgQ4bt4|Así se LOGRÓ el P...| @jeffersonsilva1089|soy de nicaragua ...|    0|2026-03-08T19:13:35Z|     

In [9]:
#Por ultimo filtramos resultados usando filter como en la tarea anterior

#comentarios con popularidad viral
df.filter( (col("popularidad") == "viral") & (col("channel_title") == "Lethal Crysis") ).show(10)

+--------------------+-------------+-----------+--------------------+---------------+--------------------+-----+--------------------+-----------+-------------------+-----------+------------+----------------+
|          channel_id|channel_title|   video_id|         video_title|         author|             comment|likes|        published_at|reply_count|longitud_caracteres|popularidad|fecha_limpia|hora_publicacion|
+--------------------+-------------+-----------+--------------------+---------------+--------------------+-----+--------------------+-----------+-------------------+-----------+------------+----------------+
|UCJCen5xxTtU7cnVH...|Lethal Crysis|kVxlVDA6zz4|OPERAR O MORIR: e...|@margaerycastle|se que youtube no...| 2211|2026-01-25T19:42:49Z|         16|                145|      viral|  2026-01-25|              19|
|UCJCen5xxTtU7cnVH...|Lethal Crysis|kVxlVDA6zz4|OPERAR O MORIR: e...|        @dreseg|a mi me extrajero...| 2017|2026-01-25T18:43:14Z|         19|                480|   

In [11]:
#comentarios con una longitud mayor a los 500
df_filtrado = df.filter(col("longitud_caracteres") > 500)
df_filtrado.orderBy(col("longitud_caracteres").desc()).show()#los ordene de mayor a menor para que sea el maximo y como desciende la longitud

+--------------------+--------------------+-----------+--------------------+--------------------+--------------------+-----+--------------------+-----------+-------------------+-----------+------------+----------------+
|          channel_id|       channel_title|   video_id|         video_title|              author|             comment|likes|        published_at|reply_count|longitud_caracteres|popularidad|fecha_limpia|hora_publicacion|
+--------------------+--------------------+-----------+--------------------+--------------------+--------------------+-----+--------------------+-----------+-------------------+-----------+------------+----------------+
|UCLyFFksH0lfVFzQZ...|Relatos Forenses ...|IrFByCz76XU|Análisis de la en...|           @manotida|muchas veces me h...|    0|2026-03-06T06:21:50Z|          0|               9337|       baja|  2026-03-06|               6|
|UCmc3SanlEKAdNyNL...|Mi Querido Mussolini|B2mvvJ-twoI|Mussolini Bailand...|@ricardofigueroaj...|papu papu papupap...|  

In [14]:
#este ultimo es como si hicieramos un query en sql donde buscamos comentarios que contengan
#la palabra 'Chile' para ver si alguien escribe desde ahi o inclusive vive ahi
#o se usa la tipica expresion 'Alchile', etc, etc.
#select * from tal.tal where trim(both ' ' comentario) like '%chile%'
#pero todo hecho aca en PySpark
df.filter((trim(col("comment")).like("%chile%"))).show()

+--------------------+----------------+-----------+--------------------+--------------------+--------------------+-----+--------------------+-----------+-------------------+-----------+------------+----------------+
|          channel_id|   channel_title|   video_id|         video_title|              author|             comment|likes|        published_at|reply_count|longitud_caracteres|popularidad|fecha_limpia|hora_publicacion|
+--------------------+----------------+-----------+--------------------+--------------------+--------------------+-----+--------------------+-----------+-------------------+-----------+------------+----------------+
|UCECJDeK0MNapZbpa...|Luisito Comunica|wupYX4W8UHA|Fui al país con l...|         @xavier_432|que chilero lugar...|    0|2026-03-05T02:08:40Z|          0|                 52|       baja|  2026-03-05|               2|
|UCECJDeK0MNapZbpa...|Luisito Comunica|wupYX4W8UHA|Fui al país con l...|@BalamasnahoficialCL|estos son los vid...|    0|2026-03-03T18:31